# FemCare - Fine-tuning LoRA/QLoRA no Colab

Notebook companion da **Fase H** do SDD `docs/sdd/ia-core/tasks.md`. Cobre IA-H1, IA-H2 e IA-H3.

**Importante:**

- Este fine-tuning ajusta **formato e linguagem clinica em portugues** para os quatro fluxos exigidos pelo desafio. Nao substitui o RAG (`fase3_orquestracao/rag_chain.py`), os guardrails (`fase4_seguranca/safety_guard.py`) nem o roteador clinico (`fase3_orquestracao/clinical_router.py`).
- Datasets `data/train.jsonl` e `data/val.jsonl` sao gerados pela Fase B a partir do **MedQuAD** ([Kaggle `pythonafroz/medquad-medical-question-answer-for-ai-research`](https://www.kaggle.com/datasets/pythonafroz/medquad-medical-question-answer-for-ai-research)) + complementos sinteticos curados em `data/synthetic/womens_health_curated.jsonl`.
- **Nao** versionamos pesos pesados; siga `docs/fine-tuning.md` para publicar o adapter (HF Hub > GitHub Release > Git LFS).

GPU recomendado: T4 (gratis no Colab) ou A100 (Colab Pro+). CPU **nao e suficiente** para este notebook; para reproducao sem GPU, rode `python fase2_finetuning/train_lora.py --dry-run`.

## 1. Setup do ambiente Colab

Clona o repositorio publico, instala as dependencias opcionais de fine-tuning e checa a GPU disponivel.

In [ ]:
# Confirma GPU; se vier vazio, troque o runtime para GPU (Runtime > Change runtime type > T4/A100).
!nvidia-smi -L

In [ ]:
import os, pathlib
REPO_URL = 'https://github.com/vinicius707/tech-challenge-fase-3-8IADT.git'
REPO_DIR = pathlib.Path('/content/tech-challenge-fase-3-8IADT')
if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
os.chdir(REPO_DIR)
!pwd

In [ ]:
# Dependencias da Fase H (incluem torch, transformers, peft, trl, bitsandbytes).
!pip install --quiet -r requirements-finetuning.txt

## 2. Gerar `data/train.jsonl` e `data/val.jsonl` (origem MedQuAD)

Os splits sao reconstruidos pela Fase B. Configure as credenciais do Kaggle no Colab (`~/.kaggle/kaggle.json`) antes de rodar o download real, ou faca upload manual do conteudo de `data/`.

In [ ]:
# Credenciais Kaggle (opcional - so se voce for baixar MedQuAD agora). Comente o bloco se preferir.
# from google.colab import files; files.upload()  # selecione kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Baixa o dataset (cacheado pelo kagglehub) e gera os splits.
!python fase1_dados/download_medquad.py --copy-to-raw
!python fase1_dados/build_dataset.py
!python fase1_dados/validate_data.py
!ls -lh data/train.jsonl data/val.jsonl

## 3. Hiperparametros (sincronizados com `train_lora.py`)

Mantemos os defaults consistentes com o script para que `metadata.json` gerado pelo notebook tenha o mesmo schema do gerado localmente. Edite com cuidado: o `validate_adapters.py` confere `sha256` dos splits.

In [ ]:
from fase2_finetuning.train_lora import LoraConfig, TrainingConfig
lora = LoraConfig()
training = TrainingConfig()
BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'  # alternativa: 'Qwen/Qwen2.5-1.5B-Instruct' em GPUs menores
OUTPUT_DIR = 'outputs/model'
print('LoRA:', lora)
print('Training:', training)
print('Base model:', BASE_MODEL)

## 4. Treino LoRA/QLoRA

Reusa a funcao `run_training` do script Python. Isso evita duplicar logica e mantem o notebook como **demonstracao** do mesmo pipeline executavel offline. Para acompanhar perda/eval, use `trainer_state.json` salvo pelo TRL em `outputs/model/`.

In [ ]:
from pathlib import Path
from fase2_finetuning.train_lora import (
    build_metadata, run_training, _validate_split, _collect_local_artifacts,
    ArtifactsExternal, METADATA_PATH, DEFAULT_TRAIN_PATH, DEFAULT_VAL_PATH,
)

train_split = _validate_split('train', Path(DEFAULT_TRAIN_PATH))
val_split = _validate_split('val', Path(DEFAULT_VAL_PATH))
results = run_training(
    base_model=BASE_MODEL,
    train_path=Path(DEFAULT_TRAIN_PATH),
    val_path=Path(DEFAULT_VAL_PATH),
    output_dir=Path(OUTPUT_DIR),
    lora=lora,
    training=training,
)
print('eval:', results)

## 5. Atualizar `outputs/model/metadata.json` (IA-H3)

Coleta tamanhos + sha256 dos artefatos locais e grava o metadata final. Esse arquivo continua sendo o **unico** rastreado pelo Git em `outputs/model/`.

In [ ]:
import json
artifacts_local = _collect_local_artifacts(Path(OUTPUT_DIR))
metadata = build_metadata(
    mode='trained',
    base_model=BASE_MODEL,
    lora=lora,
    training=training,
    train_split=train_split,
    val_split=val_split,
    artifacts_local=artifacts_local,
    artifacts_external=ArtifactsExternal(),
    training_results=results,
)
Path(METADATA_PATH).parent.mkdir(parents=True, exist_ok=True)
Path(METADATA_PATH).write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('metadata atualizado em', METADATA_PATH)

## 6. Validar adapter (IA-FT-02 / IA-H4)

Rode o gate oficial. Esperado exit code `0` e relatorio sem `error`.

In [ ]:
!python fase2_finetuning/validate_adapters.py
!cat outputs/reports/finetuning_validation.md

## 7. Publicar o adapter (IA-H5)

Pesos do LoRA NAO devem ir para o Git deste repositorio. Escolha um canal e registre em `outputs/model/metadata.json` (campo `artifacts.external`).

### 7.1 Hugging Face Hub (recomendado)

```bash
huggingface-cli login   # token com permissao write
huggingface-cli repo create femcare-llama32-lora --type model
cd outputs/model && huggingface-cli upload <org>/femcare-llama32-lora . --repo-type model
```

### 7.2 GitHub Release como fallback

```bash
zip -r adapter.zip outputs/model/
gh release create v0.1.0-lora adapter.zip --notes 'LoRA adapter Fase H'
```

Depois atualize `outputs/model/metadata.json` com o URL real e o `sha256` correto. Faca commit apenas do `metadata.json` no repositorio principal.

## 8. Carregamento posterior

Para usar o adapter no IA Core, baixe-o em `outputs/model/` (mesmo path declarado) e selecione o backend `local_lora` em `config/model_backends.yaml`:

```bash
huggingface-cli download <org>/femcare-llama32-lora --local-dir outputs/model
IA_LLM_BACKEND=local_lora python -m fase3_orquestracao.llm_backend --prompt 'teste'
```

Lembre-se: o adapter sozinho **nao** decide diagnostico nem prescricao - todos os guardrails da Fase E continuam mandando.